# Dynamic Societal Friction Simulator — TRUST-CFE

Colab-ready end-to-end notebook.

**Runtime:** Change to GPU (A100 or L4 preferred) before running Stage A.

**Pipeline:**
1. Setup (Drive mount, clone, install).
2. Download a small GDELT slice for India.
3. Build / load article corpus and factual atoms.
4. **Stage A** — train trust encoder + cleavage + hostility.
5. Build trust-weighted discourse tensor T, event tensor E, targets from ACLED.
6. **Stage B** — train friction aggregator + escalation head.
7. Evaluate (baselines, ablation).
8. Visualize.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Choose a persistent project root in Drive.
import os, sys, shutil
PROJECT_ROOT = '/content/drive/MyDrive/dsfs'
os.makedirs(PROJECT_ROOT, exist_ok=True)
# Copy the repo into Drive the first time — or git clone your fork instead.
REPO_SRC = '/content/dynamic-societal-friction-simulator'
if not os.path.isdir(REPO_SRC):
    # If you have uploaded the zip to Drive, unzip here; otherwise git clone.
    !cp -r '/content/drive/MyDrive/dynamic-societal-friction-simulator' /content/ || true
sys.path.insert(0, REPO_SRC)
print('Repo at:', REPO_SRC)

In [ ]:
!pip -q install -r /content/dynamic-societal-friction-simulator/requirements.txt

## 2. Download a small GDELT slice (India, 7 days)

In [ ]:
import datetime as dt
from src.data.gdelt_downloader import download_range

# Tiny demo window — extend later for your real experiments.
start = dt.datetime(2024, 1, 1)
end = dt.datetime(2024, 1, 7, 23, 45)
GDELT_DIR = f'{PROJECT_ROOT}/data/raw/gdelt'
download_range(start, end, GDELT_DIR, kinds=('export', 'gkg'))

## 3. ACLED India CSV

Register at [ACLED](https://acleddata.com/) (free academic access), download an India CSV covering your window, and upload to
`PROJECT_ROOT/data/raw/acled/acled_india.csv`.

In [ ]:
from src.data.acled_loader import load_acled, build_target_tensor
acled_path = f'{PROJECT_ROOT}/data/raw/acled/acled_india.csv'
acled = load_acled(acled_path)
targets = build_target_tensor(acled, horizons=(1, 2, 4))
print({k: v.shape if hasattr(v, 'shape') else v for k, v in targets.items()})

## 4. Build article corpus + factual atoms

For a full run, pull article text from GDELT's mention URLs (be mindful of rate-limits and ToS).
For the demo we'll build a minimal synthetic corpus directly from GKG rows.

In [ ]:
import pandas as pd, numpy as np, pickle, glob
from src.data.india_geo import resolve_state, iso_week_index
from src.data.factual_signals import extract_atoms, NERExtractor

gkg_files = sorted(glob.glob(f'{GDELT_DIR}/gkg/*.parquet'))
gkg = pd.concat([pd.read_parquet(p) for p in gkg_files], ignore_index=True)
print('GKG rows:', len(gkg))

# Minimal corpus — a real run should fetch article text; we use the Quotations
# / AllNames / Themes fields as a proxy text blob for demo purposes.
def _row_text(r):
    parts = [str(r.get('Quotations') or ''), str(r.get('AllNames') or ''), str(r.get('Themes') or '')]
    return ' '.join(parts)[:3000]

gkg['text'] = gkg.apply(_row_text, axis=1)
gkg['source_domain'] = gkg['SourceCommonName'].fillna('unknown').astype(str)
gkg['date'] = pd.to_datetime(gkg['DATE'].astype(str).str[:8], format='%Y%m%d', errors='coerce')
gkg = gkg.dropna(subset=['date'])
gkg['iso_week'] = gkg['date'].apply(iso_week_index)

# Approximate state: take the first IN# entry in V2Locations.
import re
def _first_state(cell):
    if not isinstance(cell, str):
        return None
    for loc in cell.split(';'):
        parts = loc.split('#')
        if len(parts) >= 5 and parts[3].strip().upper() == 'IN':
            return resolve_state(parts[4].strip())
    return None
gkg['state'] = gkg['V2Locations'].apply(_first_state)
gkg = gkg.dropna(subset=['state'])
gkg['avg_tone'] = pd.to_numeric(gkg['V2Tone'].astype(str).str.split(',').str[0], errors='coerce').fillna(0)
gkg['article_id'] = gkg['GKGRECORDID'].astype(str)

articles = gkg[['article_id','source_domain','state','date','iso_week','text','avg_tone']].reset_index(drop=True)
print('Articles:', len(articles))

In [ ]:
# Compute MuRIL CLS embeddings (cached) using the same tokenizer/backbone the
# trust encoder is built on. (The `src.models.embed` helper was removed in the
# v1 cleanup — we tokenize inline here so there is exactly one backbone path.)
import torch
from transformers import AutoModel, AutoTokenizer

_EMB_MODEL = 'google/muril-base-cased'
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_tok = AutoTokenizer.from_pretrained(_EMB_MODEL)
_mod = AutoModel.from_pretrained(_EMB_MODEL).to(_device).eval()

@torch.no_grad()
def _encode_cls(texts, batch_size=16, max_length=256):
    outs = []
    for i in range(0, len(texts), batch_size):
        enc = _tok(texts[i:i+batch_size], truncation=True, padding='max_length',
                   max_length=max_length, return_tensors='pt').to(_device)
        h = _mod(**enc).last_hidden_state[:, 0, :]
        outs.append(h.cpu().numpy())
    return np.concatenate(outs, axis=0)

embeddings = _encode_cls(articles['text'].tolist())
np.save(f'{PROJECT_ROOT}/data/processed/muril_cls.npy', embeddings)
print('Embeddings:', embeddings.shape)

In [ ]:
# Extract factual atoms (NER + numerics + triples).
ner = NERExtractor(device=0)
atoms = {}
for i, row in articles.iterrows():
    atoms[row['article_id']] = extract_atoms(row['text'], ner)
with open(f'{PROJECT_ROOT}/data/processed/atoms.pkl', 'wb') as f:
    pickle.dump(atoms, f)

In [ ]:
# Event clustering (M3).
from src.data.event_clusters import cluster_events, ClusteringConfig
ent_list = [atoms[aid].entities for aid in articles['article_id']]
articles['event_cluster_id'] = cluster_events(articles, embeddings, ent_list, ClusteringConfig())
articles = articles[articles['event_cluster_id'] >= 0]
articles.to_parquet(f'{PROJECT_ROOT}/data/processed/articles.parquet', index=False)
print('Clustered articles:', len(articles), 'clusters:', articles['event_cluster_id'].nunique())

## 5. Stage A — trust + cleavage + hostility training

In [ ]:
from src.training.train_stage_a import train_stage_a
train_stage_a(
    articles_parquet=f'{PROJECT_ROOT}/data/processed/articles.parquet',
    atoms_pickle=f'{PROJECT_ROOT}/data/processed/atoms.pkl',
    cfg_path='/content/dynamic-societal-friction-simulator/config.yaml',
    out_dir=f'{PROJECT_ROOT}/artifacts/stage_a',
    epochs=2,
)

## 6. Build E, T, targets; then Stage B

In [ ]:
# Predict per-article cleavage probs + per-cleavage hostility using trained heads.
import torch
from src.models.trust_learner import TrustEncoder, TrustConfig
from src.models.cleavage_classifier import CleavageHead, CLEAVAGES
from src.models.hostility_encoder import HostilityHead
device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = TrustEncoder(TrustConfig()); enc.load_state_dict(torch.load(f'{PROJECT_ROOT}/artifacts/stage_a/trust_encoder.pt')); enc.to(device).eval()
clh = CleavageHead(); clh.load_state_dict(torch.load(f'{PROJECT_ROOT}/artifacts/stage_a/cleavage_head.pt')); clh.to(device).eval()
hoh = HostilityHead(); hoh.load_state_dict(torch.load(f'{PROJECT_ROOT}/artifacts/stage_a/hostility_head.pt')); hoh.to(device).eval()

def predict_heads(texts, batch_size=32):
    import math
    from torch.utils.data import DataLoader
    n = len(texts); K = len(CLEAVAGES)
    probs = np.zeros((n, K), dtype=np.float32)
    hostility = np.zeros((n, K), dtype=np.float32)
    for start in range(0, n, batch_size):
        batch = texts[start:start + batch_size]
        enc_in = enc.tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors='pt').to(device)
        with torch.no_grad():
            z, s = enc(**enc_in)
            cls_for_heads = z  # trained model projects to proj_dim; head was trained on the proj
            # The stage_a trainer introduces a learned proj→hidden linear; for demo we reuse same dim.
            cls_for_heads_768 = cls_for_heads if cls_for_heads.size(-1) == 768 else torch.nn.functional.pad(cls_for_heads, (0, 768 - cls_for_heads.size(-1)))
            cl_logits = clh(cls_for_heads_768)
            probs[start:start + len(batch)] = torch.sigmoid(cl_logits).cpu().numpy()
            for ki in range(K):
                idx = torch.full((len(batch),), ki, dtype=torch.long, device=device)
                h_logits = hoh(cls_for_heads_768, idx)
                hostility[start:start + len(batch), ki] = torch.sigmoid(h_logits).cpu().numpy()
    return probs, hostility

cleavage_probs, hostility = predict_heads(articles['text'].tolist())
print(cleavage_probs.shape, hostility.shape)

In [ ]:
# Build T and E.
source_trust = pd.read_parquet(f'{PROJECT_ROOT}/artifacts/stage_a/source_trust.parquet')
from src.training.build_T_tensor import build_T_tensor
min_week = int(articles['iso_week'].min())
max_week = int(articles['iso_week'].max())
T_tensor = build_T_tensor(articles, source_trust, cleavage_probs, hostility, min_week, max_week)

from src.data.preprocessing import load_gdelt_events, attach_cleavage_from_actors, event_intensity_tensor
events = load_gdelt_events(f'{GDELT_DIR}/export')
events = attach_cleavage_from_actors(events)
E_tensor = event_intensity_tensor(events, min_week, max_week)

# Save bundle. (The optional R / relational-strain channel was removed in v1.)
y_kwargs = {}
for h in (1,2,4):
    y_kwargs[f'y_protests_h{h}'] = targets[f'y_protests_h{h}'][:, :max_week-min_week+1]
    y_kwargs[f'y_fatalities_h{h}'] = targets[f'y_fatalities_h{h}'][:, :max_week-min_week+1]
np.savez(f'{PROJECT_ROOT}/data/processed/tensors.npz', E=E_tensor, T=T_tensor, **y_kwargs)
print('E', E_tensor.shape, 'T', T_tensor.shape)

In [ ]:
from src.training.train_stage_b import train_stage_b
# In a real run set realistic week cutoffs; for this tiny demo use all-train.
train_cutoff = max_week; val_cutoff = max_week
train_stage_b(
    E=E_tensor, T=T_tensor,
    y_protests={h: y_kwargs[f'y_protests_h{h}'] for h in (1,2,4)},
    y_fatalities={h: y_kwargs[f'y_fatalities_h{h}'] for h in (1,2,4)},
    min_week=min_week, train_week_cutoff=train_cutoff, val_week_cutoff=val_cutoff,
    cfg_path='/content/dynamic-societal-friction-simulator/config.yaml',
    out_dir=f'{PROJECT_ROOT}/artifacts/stage_b', epochs=50,
)

## 7. Evaluate — baselines, metrics, ablation

In [ ]:
from src.evaluation.metrics import friction_score_accuracy, event_escalation_prediction, lead_time_auc
F_agg = np.load(f'{PROJECT_ROOT}/artifacts/stage_b/F_agg.npy')
y = y_kwargs['y_fatalities_h2']
print('FSA  :', friction_score_accuracy(F_agg, y))
print('EEP  :', event_escalation_prediction(F_agg, y))
print('LTROC:', lead_time_auc(F_agg, y))

In [ ]:
from src.evaluation.baselines import event_count_baseline, goldstein_mean_baseline, sentiment_only_baseline
b_count = event_count_baseline(events, min_week, max_week)
b_gold  = goldstein_mean_baseline(events, min_week, max_week)
b_sent  = sentiment_only_baseline(articles, hostility, min_week, max_week)
print('count   FSA:', friction_score_accuracy(b_count, y))
print('gold    FSA:', friction_score_accuracy(b_gold, y))
print('sent    FSA:', friction_score_accuracy(b_sent, y))
print('ours    FSA:', friction_score_accuracy(F_agg, y))

In [ ]:
# Ablation — same stage-B recipe but swap trust-weighted T for an unweighted one.
# Construct T_plain (τ_s ≡ 1).
import copy
plain_trust = source_trust.copy(); plain_trust['tau'] = 1.0
T_plain = build_T_tensor(articles, plain_trust, cleavage_probs, hostility, min_week, max_week)
from src.evaluation.ablation import run_ablation
run_ablation(E_tensor, T_tensor, T_plain, y_kwargs, min_week, train_cutoff, val_cutoff,
             cfg_path='/content/dynamic-societal-friction-simulator/config.yaml',
             out_dir=f'{PROJECT_ROOT}/artifacts/ablation')

## 8. Visualize

In [ ]:
from src.visualization.choropleth import plot_state_choropleth
from src.visualization.timeline import plot_state_timeline
from src.visualization.source_trust_plot import plot_source_trust
import os
os.makedirs(f'{PROJECT_ROOT}/figures', exist_ok=True)
F_k = np.load(f'{PROJECT_ROOT}/artifacts/stage_b/F_k.npy')
plot_state_choropleth(F_agg, week_idx=F_agg.shape[1]-1, out_path=f'{PROJECT_ROOT}/figures/heatmap.png')
plot_state_timeline(F_k, F_agg, state_name='Maharashtra', out_path=f'{PROJECT_ROOT}/figures/timeline_maharashtra.png')
plot_source_trust(source_trust, out_path=f'{PROJECT_ROOT}/figures/source_trust.png', top_n=30)